# Custom Byte-Level BPE Tokenizer (Rust Backend)
Training a Byte-Level BPE tokenizer using Hugging Face `tokenizers`.

In [1]:
from tokenizers import (
    Regex,
    Tokenizer,
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
)

## Step 1: Initialize an Empty BPE Model

Start with an empty `BPE` model and inspect the default JSON pipeline structure.

In [4]:
tokenizer = Tokenizer(models.BPE())
print(tokenizer.to_str(pretty=True))

{
  "version": "1.0",
  "truncation": null,
  "padding": null,
  "added_tokens": [],
  "normalizer": null,
  "pre_tokenizer": null,
  "post_processor": null,
  "decoder": null,
  "model": {
    "type": "BPE",
    "dropout": null,
    "unk_token": null,
    "continuing_subword_prefix": null,
    "end_of_word_suffix": null,
    "fuse_unk": false,
    "byte_fallback": false,
    "ignore_merges": false,
    "vocab": {},
    "merges": []
  }
}


## Step 2: Normalizer (Optional)

`NFC` standardizes equivalent Unicode byte sequences without altering characters. Avoid lowercasing or stripping whitespace, since casing and indentation matter for code and reasoning.

In [5]:
tokenizer.normalizer = normalizers.NFC()

## Step 3: Pre-Tokenizer

Splits text into chunks before BPE runs. BPE will never merge across pre-token boundaries, preventing wasteful tokens like `"dog."` or `"the\n\n"`.

- `Split(Regex(...))`: Splits contractions, letters, single digits (`\p{N}`), punctuation, and newlines.
- `ByteLevel(use_regex=False)`: Maps raw UTF-8 bytes to printable characters without applying GPT-2's built-in regex a second time.

In [6]:
SPLIT_PATTERN = (
    r"(?i:'s|'t|'re|'ve|'m|'ll|'d)"
    r"|[^\r\n\p{L}\p{N}]?\p{L}+"
    r"|\p{N}"
    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
    r"|\s*[\r\n]+"
    r"|\s+(?!\S)"
    r"|\s+"
)

tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(SPLIT_PATTERN), behavior="isolated", invert=False),
    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
])

In [7]:
print(tokenizer.pre_tokenizer.pre_tokenize_str("Hello World,    Dhiran  0042\n"))

[('Hello', (0, 5)), ('ĠWorld', (5, 11)), (',', (11, 12)), ('ĠĠĠ', (12, 15)), ('ĠDhiran', (15, 22)), ('Ġ', (22, 23)), ('Ġ', (23, 24)), ('0', (24, 25)), ('0', (25, 26)), ('4', (26, 27)), ('2', (27, 28)), ('Ċ', (28, 29))]


## Step 4: Decoder & Post-Processor

- **Decoder (`ByteLevel`)**: Reverses the byte-to-Unicode mapping (`Ġ` → space, `Ċ` → newline) back into clean UTF-8 strings.
- **Post-Processor (`ByteLevel`)**: Keeps character offsets accurate for `ByteLevel` (and is also where BOS/EOS auto-insertion can be configured if needed).

In [8]:
tokenizer.decoder = decoders.ByteLevel()
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

## Step 5: Define Special Tokens

Core sequence tokens plus reserved chat/instruction tokens for future SFT.

In [9]:
special_tokens = [
    "<|endoftext|>",
    "<|bos|>",
    "<|pad|>",
    "<|im_start|>",
    "<|im_end|>",
]

## Step 6: Configure Trainer & Train the Tokenizer

Uses a small local sample corpus to verify the pipeline (replace `sample_corpus` with a streaming HF dataset for full training). Always instantiate a fresh `BpeTrainer` before each run since it is stateful.

In [10]:
def batch_iterator(texts, batch_size=1000):
    for i in range(0, len(texts), batch_size):
        yield texts[i : i + batch_size]

sample_corpus = [
    "The quick brown fox jumps over the lazy dog. 🚀",
] * 500

trainer = trainers.BpeTrainer(
    vocab_size=32_768,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)

tokenizer.train_from_iterator(
    batch_iterator(sample_corpus),
    trainer=trainer,
    length=len(sample_corpus),
)

print("Vocab size:", tokenizer.get_vocab_size())




Vocab size: 298


## Step 7: Test Encode / Decode Round-Trip

Verify both in-corpus merges and out-of-corpus byte-level fallback decode losslessly.

In [11]:
for text in ["The quick brown fox jumps over the lazy dog. 🚀", "Hello World  🚀"]:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)
    print("Text:   ", repr(text))
    print("Tokens: ", encoded.tokens)
    print("IDs:    ", encoded.ids)
    print("Decoded:", repr(decoded))
    print()
    assert decoded == text

Text:    'The quick brown fox jumps over the lazy dog. 🚀'
Tokens:  ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.', 'ĠðŁļĢ']
IDs:     [262, 296, 290, 292, 293, 295, 285, 294, 291, 18, 297]
Decoded: 'The quick brown fox jumps over the lazy dog. 🚀'

Text:    'Hello World  🚀'
Tokens:  ['H', 'e', 'l', 'l', 'o', 'Ġ', 'W', 'o', 'r', 'l', 'd', 'Ġ', 'ĠðŁļĢ']
IDs:     [44, 73, 80, 80, 83, 225, 59, 83, 86, 80, 72, 225, 297]
Decoded: 'Hello World  🚀'

